# IMC Prosperity Log Analyzer

Parse and visualize a submission log file. Set the `LOG_PATH` and `PRODUCT` below, then run all cells.

**Outputs:**
- Per-timestamp master table: FV, best bid/ask, position, orders placed, actual fills, PnL
- Price + FV overlay plot with order markers
- Position and PnL over time
- Fill rate analysis
- Filter for 'interesting' ticks (fills, position extremes, etc)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

# === CONFIG ===
LOG_PATH = "1776550144428_302991.log"
PRODUCT = "INTARIAN_PEPPER_ROOT"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## 1. Load the log file

In [ ]:
with open(LOG_PATH) as f:
    raw = json.load(f)

print(f"Submission ID: {raw['submissionId']}")
print(f"Lambda log entries: {len(raw['logs'])}")
print(f"Trade history entries: {len(raw['tradeHistory'])}")
print(f"Activity log size: {len(raw['activitiesLog'])} chars")

## 2. Parse the activities log (market state)

One row per product per timestamp. Gives us best bid/ask/volume, mid price, running PnL.

In [ ]:
activities = pd.read_csv(StringIO(raw["activitiesLog"]), sep=";")
activities = activities[activities["product"] == PRODUCT].copy()
activities = activities.sort_values("timestamp").reset_index(drop=True)
print(f"Activity rows for {PRODUCT}: {len(activities)}")
activities.head()

## 3. Parse the lambda logs (your `self.log()` output)

Extracts whatever you printed inside `get_orders()` — FV, POS, SPREAD, and BUY/SELL entries. Supports both old format (single dict, overwriting) and new list-based format.

In [ ]:
def parse_lambda_logs(raw_logs, product):
    rows = []
    for entry in raw_logs:
        lamb = entry.get("lambdaLog", "")
        if not lamb:
            continue
        try:
            parsed = json.loads(lamb)
        except json.JSONDecodeError:
            continue

        prod = parsed.get(product, {})
        row = {"timestamp": entry["timestamp"]}

        # Copy all scalar fields
        for k, v in prod.items():
            if k in ("BUY", "SELL", "BUYS", "SELLS"):
                continue
            row[k] = v

        # Orders placed — support list or single-dict formats
        buys_field = prod.get("BUYS")
        sells_field = prod.get("SELLS")
        buy_field = prod.get("BUY")
        sell_field = prod.get("SELL")

        placed_buys = buys_field if isinstance(buys_field, list) else ([buy_field] if buy_field else [])
        placed_sells = sells_field if isinstance(sells_field, list) else ([sell_field] if sell_field else [])

        row["placed_buys"] = placed_buys
        row["placed_sells"] = placed_sells
        row["n_buys_placed"] = len(placed_buys)
        row["n_sells_placed"] = len(placed_sells)
        row["total_buy_vol"] = sum(b.get("v", 0) for b in placed_buys)
        row["total_sell_vol"] = sum(s.get("v", 0) for s in placed_sells)

        rows.append(row)

    return pd.DataFrame(rows)

lambda_df = parse_lambda_logs(raw["logs"], PRODUCT)
print(f"Lambda log rows: {len(lambda_df)}")
lambda_df.head()

## 4. Parse the trade history (actual fills)

`SUBMISSION` as buyer = we bought. `SUBMISSION` as seller = we sold. Others = bot-to-bot trades.

In [ ]:
trades = pd.DataFrame(raw["tradeHistory"])
trades = trades[trades["symbol"] == PRODUCT].copy()

def classify(row):
    if row["buyer"] == "SUBMISSION":
        return "our_buy"
    elif row["seller"] == "SUBMISSION":
        return "our_sell"
    else:
        return "other"

trades["kind"] = trades.apply(classify, axis=1)
trades["signed_qty"] = trades.apply(
    lambda r: r["quantity"] if r["kind"] == "our_buy" else (-r["quantity"] if r["kind"] == "our_sell" else 0),
    axis=1
)
print(f"Total trades: {len(trades)}")
print(trades["kind"].value_counts())
trades.head()

In [ ]:
# Aggregate our fills per timestamp
our_trades = trades[trades["kind"].isin(["our_buy", "our_sell"])].copy()

fills_per_ts = (
    our_trades.groupby(["timestamp", "kind"])
    .agg(price_avg=("price", "mean"), qty_total=("quantity", "sum"))
    .unstack("kind", fill_value=0)
)
fills_per_ts.columns = [f"{a}_{b}" for a, b in fills_per_ts.columns]
fills_per_ts = fills_per_ts.reset_index()
print(f"Timestamps with fills: {len(fills_per_ts)}")
fills_per_ts.head()

## 5. Build the master table

One row per timestamp: market state + your logged values + placed orders + actual fills.

In [ ]:
df = activities[["timestamp", "bid_price_1", "bid_volume_1", "ask_price_1", "ask_volume_1",
                 "mid_price", "profit_and_loss"]].copy()
df = df.rename(columns={"bid_price_1": "best_bid", "ask_price_1": "best_ask",
                        "bid_volume_1": "bid_vol", "ask_volume_1": "ask_vol",
                        "profit_and_loss": "pnl"})

df = df.merge(lambda_df, on="timestamp", how="left")
df = df.merge(fills_per_ts, on="timestamp", how="left")

for col in ["qty_total_our_buy", "qty_total_our_sell"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df["spread"] = df["best_ask"] - df["best_bid"]

print(f"Master table shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## 6. Main analysis table

Key fields per timestamp. Orders shown as `volume@price`.

In [ ]:
def format_orders(orders):
    if not isinstance(orders, list) or len(orders) == 0:
        return ""
    return ", ".join(f"{o.get('v', '?')}@{o.get('p', '?')}" for o in orders)

display_df = df.copy()
display_df["buys"] = display_df["placed_buys"].apply(format_orders)
display_df["sells"] = display_df["placed_sells"].apply(format_orders)

available = set(display_df.columns)
want = ["timestamp", "best_bid", "best_ask", "spread", "mid_price"]
for opt in ["FV", "POS", "ZSCORE", "SKEW_SHIFT"]:
    if opt in available:
        want.append(opt)
want += ["buys", "sells"]
if "qty_total_our_buy" in available:
    want += ["qty_total_our_buy", "qty_total_our_sell"]
want += ["pnl"]

final_cols = [c for c in want if c in available]
display_df[final_cols].head(30)

In [ ]:
# Export to CSV for offline analysis
df.to_csv("log_analysis.csv", index=False)
print("Saved to log_analysis.csv")

## 7. Price, FV, and fills overlay

Mid price + best bid/ask + your FV estimate. Green triangles = our buys. Red triangles = our sells.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(df["timestamp"], df["mid_price"], color="gray", alpha=0.5, linewidth=0.8, label="Mid")
ax.plot(df["timestamp"], df["best_bid"], color="green", alpha=0.4, linewidth=0.5, label="Best bid")
ax.plot(df["timestamp"], df["best_ask"], color="red", alpha=0.4, linewidth=0.5, label="Best ask")

if "FV" in df.columns:
    ax.plot(df["timestamp"], df["FV"], color="blue", linewidth=1.2, label="FV (your estimate)")

our_buys = our_trades[our_trades["kind"] == "our_buy"]
our_sells = our_trades[our_trades["kind"] == "our_sell"]
ax.scatter(our_buys["timestamp"], our_buys["price"], marker="^", color="lime",
           s=30, alpha=0.8, label=f"Our buys ({len(our_buys)})", zorder=5, edgecolors="black", linewidths=0.3)
ax.scatter(our_sells["timestamp"], our_sells["price"], marker="v", color="red",
           s=30, alpha=0.8, label=f"Our sells ({len(our_sells)})", zorder=5, edgecolors="black", linewidths=0.3)

ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.set_title(f"{PRODUCT} \u2014 market state, FV, and our fills")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Position and PnL over time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={"height_ratios": [1, 1]})

our_trades_sorted = our_trades.sort_values("timestamp").copy()
our_trades_sorted["running_pos"] = our_trades_sorted["signed_qty"].cumsum()

ax1.step(our_trades_sorted["timestamp"], our_trades_sorted["running_pos"], where="post",
         color="blue", linewidth=1.2, label="Position (from fills)")
if "POS" in df.columns:
    ax1.plot(df["timestamp"], df["POS"], color="orange", alpha=0.6, linewidth=0.8, label="POS (your log)")

ax1.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax1.axhline(80, color="red", linestyle=":", alpha=0.5, label="Position limit")
ax1.axhline(-80, color="red", linestyle=":", alpha=0.5)
ax1.set_ylabel("Position")
ax1.legend(loc="best", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title(f"{PRODUCT} \u2014 Position & PnL")

ax2.plot(df["timestamp"], df["pnl"], color="green", linewidth=1.2)
ax2.fill_between(df["timestamp"], 0, df["pnl"], where=df["pnl"]>=0, color="green", alpha=0.15)
ax2.fill_between(df["timestamp"], 0, df["pnl"], where=df["pnl"]<0, color="red", alpha=0.15)
ax2.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax2.set_ylabel("Running PnL")
ax2.set_xlabel("Timestamp")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final PnL:     {df['pnl'].iloc[-1]:.2f}")
print(f"Max drawdown:  {(df['pnl'] - df['pnl'].cummax()).min():.2f}")
print(f"Total trades:  {len(our_trades)} ({len(our_buys)} buys, {len(our_sells)} sells)")

## 9. Fill rate analysis

For each placed order, did we get filled?

In [ ]:
fill_df = df[["timestamp", "total_buy_vol", "total_sell_vol"]].copy()
if "qty_total_our_buy" in df.columns:
    fill_df["filled_buy_vol"] = df["qty_total_our_buy"]
    fill_df["filled_sell_vol"] = df["qty_total_our_sell"]
else:
    fill_df["filled_buy_vol"] = 0
    fill_df["filled_sell_vol"] = 0

fill_df["buy_fill_rate"] = np.where(fill_df["total_buy_vol"] > 0,
                                     fill_df["filled_buy_vol"] / fill_df["total_buy_vol"], np.nan)
fill_df["sell_fill_rate"] = np.where(fill_df["total_sell_vol"] > 0,
                                      fill_df["filled_sell_vol"] / fill_df["total_sell_vol"], np.nan)

total_placed_buy = fill_df['total_buy_vol'].sum()
total_placed_sell = fill_df['total_sell_vol'].sum()
total_filled_buy = fill_df['filled_buy_vol'].sum()
total_filled_sell = fill_df['filled_sell_vol'].sum()

print("Overall fill rates:")
print(f"  Buy side:  placed {total_placed_buy} units, filled {total_filled_buy} "
      f"({total_filled_buy / max(total_placed_buy, 1) * 100:.1f}%)")
print(f"  Sell side: placed {total_placed_sell} units, filled {total_filled_sell} "
      f"({total_filled_sell / max(total_placed_sell, 1) * 100:.1f}%)")

print("\nFill rate by side (per-tick avg):")
print(f"  Buy:  {fill_df['buy_fill_rate'].mean() * 100:.1f}% of placed volume per tick")
print(f"  Sell: {fill_df['sell_fill_rate'].mean() * 100:.1f}% of placed volume per tick")

## 10. Drill into specific ticks

Default filter: only ticks where we had fills. Edit the filter to inspect other interesting moments (position at limit, high z-score, etc).

In [ ]:
interesting = df[(df.get("qty_total_our_buy", 0) > 0) | (df.get("qty_total_our_sell", 0) > 0)].copy()
interesting["buys"] = interesting["placed_buys"].apply(format_orders)
interesting["sells"] = interesting["placed_sells"].apply(format_orders)

show_cols = ["timestamp", "best_bid", "best_ask"]
if "FV" in interesting.columns: show_cols.append("FV")
if "POS" in interesting.columns: show_cols.append("POS")
show_cols += ["buys", "sells"]
if "qty_total_our_buy" in interesting.columns: show_cols += ["qty_total_our_buy", "qty_total_our_sell"]
show_cols.append("pnl")

show_cols = [c for c in show_cols if c in interesting.columns]
print(f"Ticks with fills: {len(interesting)}")
interesting[show_cols].head(50)

### Other useful filters

Uncomment any of these to explore different slices:

In [ ]:
# At position limit
# df[df.get("POS", pd.Series(0)).abs() >= 80].head(20)

# Ticks where FV diverged from mid
# if "FV" in df.columns:
#     df["fv_resid"] = df["FV"] - df["mid_price"]
#     df[df["fv_resid"].abs() > 2].head(20)

# Largest single-tick PnL changes (worst and best)
# df["pnl_delta"] = df["pnl"].diff()
# print("Biggest PnL drops:"); display(df.nsmallest(10, "pnl_delta"))
# print("Biggest PnL gains:"); display(df.nlargest(10, "pnl_delta"))

# Narrow spread events (potential taking opportunities)
# df[df["spread"] < 10].head(20)